In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# from pydantic_ai import Agent
# from pydantic_ai.models.openai import OpenAIChatModel

# # Define the AI agent
# agent = Agent(
#     model=OpenAIChatModel("deepseek-v3"),
#     system_message="You are an AI assistant that retrieves code from GitHub."
# )
#  # Test the AI agent
# response = agent.run("Find the README file in this repo: https://github.com/example/repo")
# print(response)

In [ ]:
# from pydantic_ai import Agent
# from pydantic_ai.models.groq import GroqModel

# model = GroqModel('openai/gpt-oss-120b')
# agent = Agent(model)

# response = await agent.run("Find the README file in this repo: https://github.com/Sourav692/AI_Enginnering_Projects")
# print(response.output)

In [ ]:
repo_url = "https://github.com/Sourav692/Agents_with_LangGraph_Demystified"

In [ ]:
import os
import requests

def get_repo_structure(repo_url):
    """Fetches the structure of a GitHub repository."""
    headers = {"Authorization": f"token {os.getenv('GITHUB_ACCESS_TOKEN')}"}
    repo_parts = repo_url.rstrip('/').split('/')
    owner = repo_parts[-2]
    repo = repo_parts[-1]
    print(owner, repo)
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/"
    response = requests.get(api_url, headers=headers)
    if response.status_code == 200:
        return [item["path"] for item in response.json()]
    else:
        return f"Error: {response.status_code}, {response.text}"

# Test it
print(get_repo_structure(repo_url))

In [ ]:
def get_file_contents(repo_url, file_path):
    """Fetches the contents of a specific file from a GitHub repository."""
    headers = {"Authorization": f"token {os.getenv('GITHUB_ACCESS_TOKEN')}"}
    api_url = f"https://api.github.com/repos/{repo_url.split('/')[-2]}/{repo_url.split('/')[-1]}/contents/{file_path}"
    response = requests.get(api_url, headers=headers)
    if response.status_code == 200:
        import base64
        file_content = base64.b64decode(response.json()["content"]).decode("utf-8")
        return file_content
    else:
        return f"Error: {response.status_code}, {response.text}"
 # Test it
file_path = "README.md"
print(get_file_contents(repo_url, file_path))

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

# Define the AI agent
agent = Agent(
    model=GroqModel('openai/gpt-oss-120b'),
    system_prompt="""You are an AI assistant trained to analyze code from GitHub. Only Analyze the code from the given Repo URL and provide the answer in a concise manner. 
    Dont provide any other information if you are not sure about the answer"""
)


async def analyze_file(repo_url: str, file_path: str):
    """Uses the AI model to analyze the contents of a GitHub file."""
    file_content = get_file_contents(repo_url, file_path)
    # if "Error" in file_content:
    #     return file_content
    result = await agent.run(f"Analyze this code and explain what it does:\n\n{file_content}")
    return result.output


# Test it (Jupyter supports top-level await)
print(await analyze_file(repo_url, "README.md"))

In [ ]:
# def select_model(task: str) -> str:
#     """Dynamically selects the best AI model based on the task."""
#     if "code" in task.lower():
#         return "deepseek-v3"  # Best for code-related queries
#     elif "write" in task.lower():
#         return "gpt-4o"  # More creative writing and explanations
#     elif "analyze" in task.lower():
#         return "claude-3"  # Best for logical breakdowns
#     else:
#         return "gemini-pro"  # Balanced for general use


# # Example usage
# task_description = "Analyze this Python function for security flaws."
# model_to_use = select_model(task_description)
# print(f"Using model: {model_to_use}")

In [ ]:
class Memory:
    """A simple short-term memory for AI agents."""

    def __init__(self, max_size: int = 5):
        self.history: list[dict[str, str]] = []
        self.max_size = max_size

    def add_message(self, user_input: str, ai_response: str) -> None:
        """Stores the last few interactions."""
        self.history.append({"user": user_input, "ai": ai_response})
        if len(self.history) > self.max_size:
            self.history.pop(0)  # Remove the oldest memory

    def get_memory(self) -> str:
        """Formats memory into a readable string."""
        return "\n".join([f"User: {m['user']}\nAI: {m['ai']}" for m in self.history])


# Example usage
memory = Memory()
memory.add_message("Describe this repo", "This is a Python project with multiple modules.")
print(memory.get_memory())

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.groq import GroqModel

# Define the AI agent
agent = Agent(
    model=GroqModel('openai/gpt-oss-120b'),
    system_prompt="""You are an AI assistant trained to analyze code from GitHub. Only Analyze the code from the given Repo URL and provide the answer in a concise manner. 
    Dont provide any other information if you are not sure about the answer."""
)

memory = Memory()


async def chat_with_memory(user_input: str):
    """Enhances the AI agent with memory."""
    # past_conversations = memory.get_memory()
    # full_prompt = f"{past_conversations}\n\nUser: {user_input}\nAI:"
    result = await agent.run(user_input)
    ai_response = result.output
    # memory.add_message(user_input, ai_response)
    return ai_response

In [ ]:
# Example conversation
print(await chat_with_memory(f"What is the main purpose of this repository?{repo_url}"))

In [ ]:
print(await chat_with_memory(f"Show me the summary of helpers/utils.py file in this repo {repo_url}"))

In [ ]:
import json


class PersistentMemory:
    """A memory system that stores AI interactions in a JSON file."""

    def __init__(self, filename: str = "memory.json"):
        self.filename = filename
        self.load_memory()

    def load_memory(self) -> None:
        """Loads memory from file."""
        try:
            with open(self.filename, "r") as f:
                self.history = json.load(f)
        except FileNotFoundError:
            self.history = []

    def save_memory(self) -> None:
        """Saves memory to file."""
        with open(self.filename, "w") as f:
            json.dump(self.history, f, indent=4)

    def add_message(self, user_input: str, ai_response: str) -> None:
        """Stores interaction and persists it."""
        self.history.append({"user": user_input, "ai": ai_response})
        self.save_memory()

    def get_memory(self) -> str:
        """Formats memory for AI input (last 5 exchanges)."""
        return "\n".join([f"User: {m['user']}\nAI: {m['ai']}" for m in self.history[-5:]])


# Example usage
memory = PersistentMemory()
memory.add_message("Explain the README file", "The README provides an overview of the project.")
print(memory.get_memory())

In [ ]:
# --- Agent that traverses the whole repo and answers with code snippets ---
import base64
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext
from pydantic_ai.models.groq import GroqModel


@dataclass
class RepoContext:
    """Context passed to the agent - contains repo URL for file operations."""
    repo_url: str


def _get_owner_repo(repo_url: str) -> tuple[str, str]:
    """Extract owner and repo name from GitHub URL."""
    parts = repo_url.rstrip("/").split("/")
    return parts[-2], parts[-1]


def _get_full_repo_tree(repo_url: str) -> list[str]:
    """Get all file paths in the repo (recursive) using GitHub Git Tree API."""
    owner, repo = _get_owner_repo(repo_url)
    headers = {"Authorization": f"token {os.getenv('GITHUB_ACCESS_TOKEN')}"}
    # Get default branch and latest commit
    repo_resp = requests.get(f"https://api.github.com/repos/{owner}/{repo}", headers=headers)
    if repo_resp.status_code != 200:
        return [f"Error: {repo_resp.status_code}"]
    default_branch = repo_resp.json().get("default_branch", "main")
    commit_resp = requests.get(
        f"https://api.github.com/repos/{owner}/{repo}/commits/{default_branch}",
        headers=headers,
    )
    if commit_resp.status_code != 200:
        return [f"Error: {commit_resp.status_code}"]
    tree_sha = commit_resp.json()["commit"]["tree"]["sha"]
    tree_resp = requests.get(
        f"https://api.github.com/repos/{owner}/{repo}/git/trees/{tree_sha}?recursive=1",
        headers=headers,
    )
    if tree_resp.status_code != 200:
        return [f"Error: {tree_resp.status_code}"]
    tree = tree_resp.json()
    return [item["path"] for item in tree.get("tree", []) if item["type"] == "blob"]


def _list_repo_contents(repo_url: str, path: str = "") -> list[str]:
    """List files and directories at a given path in the repo."""
    owner, repo = _get_owner_repo(repo_url)
    headers = {"Authorization": f"token {os.getenv('GITHUB_ACCESS_TOKEN')}"}
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}"
    response = requests.get(api_url, headers=headers)
    if response.status_code != 200:
        return [f"Error: {response.status_code} - {response.text}"]
    items = response.json()
    return [item["path"] for item in items]


def _read_file_content(repo_url: str, file_path: str) -> str:
    """Read the raw content of a file from anywhere in the repo."""
    owner, repo = _get_owner_repo(repo_url)
    headers = {"Authorization": f"token {os.getenv('GITHUB_ACCESS_TOKEN')}"}
    api_url = f"https://api.github.com/repos/{owner}/{repo}/contents/{file_path}"
    response = requests.get(api_url, headers=headers)
    if response.status_code != 200:
        return f"Error: Could not fetch file ({response.status_code}). {response.text}"
    data = response.json()
    if data.get("type") != "file":
        return f"Error: '{file_path}' is not a file (might be a directory)."
    content_b64 = data.get("content", "")
    return base64.b64decode(content_b64).decode("utf-8")


# --- Agent tools (inject repo_url from RunContext) ---
# Note: get_full_repo_tree removed - we pre-inject it into the prompt to avoid LLM null-args validation errors
def list_repo_contents(ctx: RunContext[RepoContext], path: str = "") -> str:
    """List files and directories at a path. path: relative path, empty string for repo root."""
    result = _list_repo_contents(ctx.deps.repo_url, path)
    return "\n".join(result) if isinstance(result, list) else str(result)


def read_file_content(ctx: RunContext[RepoContext], file_path: str) -> str:
    """Read the full content of a file. file_path: e.g. 'helpers/utils.py' or 'README.md'."""
    return _read_file_content(ctx.deps.repo_url, file_path)


file_reading_agent = Agent(
    model=GroqModel("moonshotai/kimi-k2-instruct-0905"),
    deps_type=RepoContext,
    system_prompt="""You are an AI assistant that analyzes GitHub repositories and answers questions using actual code from the repo.

The user message will include a list of ALL files in the repo. Use that to identify relevant files, then call read_file_content(file_path) to fetch their contents.

WORKFLOW:
1. Review the file list provided - identify which files are relevant (e.g. .py for code, README for overview)
2. Call read_file_content(file_path) for each relevant file
3. Answer using CODE SNIPPETS - format in markdown code blocks, cite file paths

When answering:
- Always cite the file path when showing code (e.g. "From helpers/utils.py:")
- Use ```python or ```markdown code blocks for snippets
- Be concise but thorough - include the actual code that answers the question""",
    tools=[list_repo_contents, read_file_content],
)


async def ask_agent_about_repo(repo_url: str, question: str) -> str:
    """Ask a question about the repo. Pre-fetches repo tree and injects into prompt to avoid tool validation issues."""
    repo_tree = _get_full_repo_tree(repo_url)
    file_list = "\n".join(repo_tree) if isinstance(repo_tree, list) else str(repo_tree)
    enriched_prompt = f"""All files in this repo:
{file_list}

---
User question: {question}"""
    result = await file_reading_agent.run(enriched_prompt, deps=RepoContext(repo_url=repo_url))
    return result.output


# Example: ask a question - agent explores the repo and answers with code
print(await ask_agent_about_repo(repo_url, "How does this repo initialize LLMs? Show me the relevant code snippets."))

In [ ]:
# Example: ask a question - agent explores the whole repo and answers with code snippets
print(await ask_agent_about_repo(repo_url, "How does this repo implement RAG? Show me the key code snippets."))

In [2]:
"""
LLM Utility Functions for LangGraph Applications
=================================================

This module provides helper functions to create and configure Large Language Model (LLM) 
instances from different providers (OpenAI and Groq). These utilities are used throughout 
the LangGraph tutorials to maintain consistent LLM initialization.

Why use utility functions for LLM initialization?
-------------------------------------------------
1. **Centralized Configuration**: All LLM settings are managed in one place
2. **Easy Provider Switching**: Switch between OpenAI and Groq with a single function call
3. **Default Parameters**: Sensible defaults (like temperature=0 for deterministic outputs)
4. **Code Reusability**: Avoid repeating initialization code across notebooks

Dependencies:
- langchain_groq: LangChain integration for Groq's fast inference API
- langchain.chat_models: LangChain's chat model abstractions for OpenAI
- python-dotenv: For loading API keys from .env files
"""

# ============================================================================
# IMPORTS
# ============================================================================

# ChatGroq: LangChain wrapper for Groq's API - known for extremely fast inference
# from langchain_groq import ChatGroq

# ChatOpenAI: LangChain wrapper for OpenAI's Chat API (GPT-3.5, GPT-4, etc.)
# from langchain.chat_models import ChatOpenAI
from langchain_openai import ChatOpenAI
from databricks_langchain import ChatDatabricks

# python-dotenv: Loads environment variables from a .env file
# This keeps API keys secure and out of source code
from dotenv import load_dotenv

# os module: Used to access environment variables after they're loaded
import os

# ============================================================================
# ENVIRONMENT SETUP
# ============================================================================

# Load environment variables from .env file in the project root
# Your .env file should contain:
#   OPENAI_API_KEY=your_openai_api_key_here
#   GROQ_API_KEY=your_groq_api_key_here
# 
# SECURITY NOTE: Never commit .env files to version control!
load_dotenv()


# ============================================================================
# LLM FACTORY FUNCTIONS
# ============================================================================

def get_openai_llm(model_name: str = "gpt-4o-mini", temperature: float = 0):
    """
    Create and return an OpenAI Chat LLM instance.
    
    This function creates a ChatOpenAI instance configured for use with LangGraph.
    OpenAI models are known for their strong reasoning and instruction-following capabilities.
    
    Parameters:
    -----------
    model_name : str, default="gpt-4o-mini"
        The OpenAI model to use. Common options:
        - "gpt-4o-mini": Fast, cost-effective, good for most tasks (recommended for learning)
        - "gpt-4o": Most capable model, best for complex reasoning
        - "gpt-4-turbo": Balance of capability and speed
        - "gpt-3.5-turbo": Fastest and cheapest, good for simple tasks
    
    temperature : float, default=0
        Controls randomness in responses (0.0 to 2.0):
        - 0: Deterministic, consistent outputs (best for agents & tools)
        - 0.7: Balanced creativity and consistency
        - 1.0+: More creative/random outputs
    
    Returns:
    --------
    ChatOpenAI
        A configured LangChain ChatOpenAI instance ready for use in chains/agents
    
    Example:
    --------
    >>> llm = get_openai_llm()  # Uses defaults
    >>> llm = get_openai_llm(model_name="gpt-4o", temperature=0.5)
    >>> response = llm.invoke("Hello, world!")
    
    Note:
    -----
    Requires OPENAI_API_KEY environment variable to be set.
    """
    return ChatOpenAI(
        model=model_name,
        temperature=temperature
    )


def get_groq_llm(model_name: str = "llama-3.3-70b-versatile", temperature: float = 0):
    """
    Create and return a Groq Chat LLM instance.
    
    Groq provides extremely fast inference using their custom LPU (Language Processing Unit)
    hardware. This makes it excellent for applications requiring low latency responses.
    
    Parameters:
    -----------
    model_name : str, default="llama-3.3-70b-versatile"
        The model to use via Groq's API. Popular options:
        - "llama-3.3-70b-versatile": Latest Llama 3.3, great balance of speed & quality
        - "llama-3.1-70b-versatile": Llama 3.1 70B, excellent for complex tasks
        - "llama-3.1-8b-instant": Smaller, faster model for simpler tasks
        - "mixtral-8x7b-32768": Mixtral model with 32k context window
        - "gemma2-9b-it": Google's Gemma 2 model
    
    temperature : float, default=0
        Controls randomness in responses (0.0 to 2.0):
        - 0: Deterministic outputs (recommended for agents & tool use)
        - Higher values: More creative/varied responses
    
    Returns:
    --------
    ChatGroq
        A configured LangChain ChatGroq instance ready for use in chains/agents
    
    Example:
    --------
    >>> llm = get_groq_llm()  # Uses defaults
    >>> llm = get_groq_llm(model_name="llama-3.1-8b-instant", temperature=0.3)
    >>> response = llm.invoke("Explain quantum computing")
    
    Why use Groq?
    -------------
    - Extremely fast inference (often 10x faster than traditional cloud providers)
    - Cost-effective for high-volume applications
    - Supports popular open-source models (Llama, Mixtral, etc.)
    
    Note:
    -----
    Requires GROQ_API_KEY environment variable to be set.
    Get your free API key at: https://console.groq.com/

    ## Use openai/gpt-oss-120b if you want to use OpenAI's models
    ## Use moonshotai/kimi-k2-instruct-0905 if you want to use Moonshot's models
    """
    # return ChatGroq(
    #     model=model_name,
    #     temperature=temperature
    # )


def get_databricks_llm(model_name: str = "databricks-gpt-5-2", temperature: float = 0.1):
    """
    Create and return a Databricks Chat LLM instance.
    """
    return ChatDatabricks(
        endpoint=model_name,
        temperature=temperature
    )

In [3]:
# llm = get_databricks_llm("databricks-claude-opus-4-6")

In [8]:
# ============================================================================
# LangGraph Agent — same capability as the pydantic_ai version above
# Traverses an entire GitHub repo and answers questions with code snippets
# ============================================================================

import base64, os, requests
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
# from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent


# ── Helpers ──────────────────────────────────────────────────────────────────

def _owner_repo(repo_url: str) -> tuple[str, str]:
    parts = repo_url.rstrip("/").split("/")
    return parts[-2], parts[-1]


def _headers() -> dict:
    return {"Authorization": f"token {os.getenv('GITHUB_ACCESS_TOKEN')}"}


# File extensions we care about (keeps the tree small & focused)
_CODE_EXTS = {
    ".py", ".md", ".txt", ".yml", ".yaml", ".json", ".toml",
    ".cfg", ".ini", ".sh", ".env", ".rst", ".csv",
}


def _full_repo_tree(repo_url: str, subfolder: str = "") -> list[str]:
    """Return code-relevant file paths in the repo (recursive, filtered).
    If subfolder is set, only return files under that directory."""
    owner, repo = _owner_repo(repo_url)
    h = _headers()
    r = requests.get(f"https://api.github.com/repos/{owner}/{repo}", headers=h)
    if r.status_code != 200:
        return [f"Error: {r.status_code}"]
    branch = r.json().get("default_branch", "main")
    r2 = requests.get(
        f"https://api.github.com/repos/{owner}/{repo}/commits/{branch}", headers=h
    )
    if r2.status_code != 200:
        return [f"Error: {r2.status_code}"]
    sha = r2.json()["commit"]["tree"]["sha"]
    r3 = requests.get(
        f"https://api.github.com/repos/{owner}/{repo}/git/trees/{sha}?recursive=1",
        headers=h,
    )
    if r3.status_code != 200:
        return [f"Error: {r3.status_code}"]
    all_files = [i["path"] for i in r3.json().get("tree", []) if i["type"] == "blob"]
    # Scope to subfolder if provided
    if subfolder:
        prefix = subfolder.rstrip("/") + "/"
        all_files = [f for f in all_files if f.startswith(prefix)]
    # Filter to code-relevant extensions only (skip .ipynb, images, etc.)
    filtered = [
        f for f in all_files
        if any(f.endswith(ext) for ext in _CODE_EXTS)
    ]
    return filtered


# ── LangChain Tools ──────────────────────────────────────────────────────────
# Bound to repo_url via closure

def make_tools(repo_url: str):
    """Create tools bound to a specific repo_url."""

    @tool
    def list_repo_contents(path: str) -> str:
        """List files and directories at a given path in the repo.
        Use path="" for root, or e.g. "helpers" for subdirectories."""
        owner, repo = _owner_repo(repo_url)
        url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}"
        r = requests.get(url, headers=_headers())
        if r.status_code != 200:
            return f"Error: {r.status_code} - {r.text}"
        return "\n".join(item["path"] for item in r.json())

    @tool
    def read_file_content(file_path: str) -> str:
        """Read the full content of a file anywhere in the repo.
        file_path examples: 'helpers/utils.py', 'README.md'."""
        owner, repo = _owner_repo(repo_url)
        url = f"https://api.github.com/repos/{owner}/{repo}/contents/{file_path}"
        r = requests.get(url, headers=_headers())
        if r.status_code != 200:
            return f"Error: Could not fetch '{file_path}' ({r.status_code})."
        data = r.json()
        if data.get("type") != "file":
            return f"Error: '{file_path}' is a directory, not a file."
        content = base64.b64decode(data.get("content", "")).decode("utf-8")
        # Truncate very large files to avoid token limits
        if len(content) > 8000:
            content = content[:8000] + "\n\n... [TRUNCATED — file too large] ..."
        return content

    return [list_repo_contents, read_file_content]


# ── Build LangGraph ReAct Agent ──────────────────────────────────────────────

def build_agent(repo_url: str, subfolder: str = ""):
    """Build a LangGraph ReAct agent bound to a specific repo (optionally scoped to a subfolder)."""

    tools = make_tools(repo_url)
    # Use OpenAI for the agent — it has reliable structured tool calling support.
    # (ChatDatabricks endpoints may emit text-based tool calls that fail validation.)
    llm = get_databricks_llm("databricks-claude-opus-4-6")

    # Pre-fetch filtered repo tree (scoped to subfolder if provided)
    repo_tree = _full_repo_tree(repo_url, subfolder=subfolder)
    file_list = "\n".join(repo_tree) if isinstance(repo_tree, list) else str(repo_tree)

    scope_desc = f"(scoped to '{subfolder}')" if subfolder else "(entire repo)"

    system_prompt = f"""You are an AI assistant that analyzes GitHub repositories and answers questions using actual code.

Here are the code-relevant files {scope_desc}:
{file_list}

WORKFLOW:
1. Pick files relevant to the user's question from the list above
2. Call read_file_content(file_path) for each relevant file (max 3-4 files)
3. Answer with CODE SNIPPETS in markdown code blocks, citing file paths

Rules:
- Always cite the file path (e.g. "From helpers/utils.py:")
- Use ```python code blocks for snippets
- Be concise — include only the code that answers the question
- If you need to browse a directory, use list_repo_contents(path)"""

    # create_react_agent handles the tool-calling loop robustly
    return create_react_agent(
        model=llm,
        tools=tools,
        prompt=system_prompt,
    )


async def ask_langgraph_agent(repo_url: str, question: str, subfolder: str = "") -> str:
    """Ask the LangGraph agent a question about a repo.
    Set subfolder to scope the agent to only files under that directory."""
    agent = build_agent(repo_url, subfolder=subfolder)
    result = await agent.ainvoke(
        {"messages": [HumanMessage(content=question)]}
    )
    return result["messages"][-1].content


# ── Test it — scoped to "6. Agentic RAG Essential" ──────────────────────────
repo_url = "https://github.com/Sourav692/Agents_with_LangGraph_Demystified"
print(await ask_langgraph_agent(
    repo_url,
    "How does the Agentic RAG work in this folder? Show me the key code snippets.",
    subfolder="6. Agentic RAG Essential",
))

/var/folders/b_/0hx_r16d2v712fg4k7y_rh140000gp/T/ipykernel_55635/807585539.py:138: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  return create_react_agent(


I'm sorry, but I'm unable to access the repository due to an authentication error (401 — "Bad credentials"). The API credentials for this repo appear to be invalid or expired, so I cannot browse or read any files.

**Here's what I can do to help:**

1. **If you can share the file contents directly** (paste them here), I'll happily analyze the Agentic RAG code and explain the key snippets.

2. **In the meantime, here's a general overview of how Agentic RAG typically works:**

### Agentic RAG — General Architecture

```python
# 1. AGENT SETUP — An LLM-based agent is given tools, including a retriever tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever,
    name="knowledge_base_search",
    description="Search the knowledge base for relevant documents"
)

# 2. The agent DECIDES whether to retrieve, re-retrieve, or answer directly
agent = create_tool_cal

In [ ]:
print(await ask_langgraph_agent(
    repo_url,
    "Explain Agentic RAG using the code snippets in this folder",
    subfolder="6. Agentic RAG Essential",
))

In [ ]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())